In [ ]:
!pip install -q langchain-huggingface langchain-qdrant langchain-community sentence-transformers rank_bm25 nltk numpy pandas matplotlib langchain_groq

In [2]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [4]:
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder
from nltk.tokenize import word_tokenize
from langchain_core.prompts import ChatPromptTemplate

Setting configs

In [5]:
QDRANT_URL = "https://f398f5e8-2ee5-4037-9590-07dbf4677382.sa-east-1-0.aws.cloud.qdrant.io"
QDRANT_API_KEY = ""

EMBED_MODEL = "abhinand/MedEmbed-base-v0.1"
COLLECTION_NAME = "ada_model_medembed_base_v0.1"
RERANKER_MODEL = "BAAI/bge-reranker-base"


QDRANT_TOP = 10
BM25_TOP = 10
FINAL_TOP = 5

In [6]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [8]:
EVAL_PATH = Path("/content/eval_queries_multi_chunk.json")
CHUNKS_PATH = Path("/content/drive/MyDrive/processed_chunks")

with open(EVAL_PATH) as f:
    eval_queries = json.load(f)

processed_chunks = []
for file_path in CHUNKS_PATH.glob("*.json"):
    with open(file_path, "r") as f:
        processed_chunks.extend(json.load(f))

In [9]:
print(f"Loaded {len(eval_queries)} queries")
print(f"Loaded {len(processed_chunks)} chunks for BM25 index")

Loaded 30 queries
Loaded 2632 chunks for BM25 index


Metrics

In [10]:
def precision_at_k(retrieved: list, relevant: list, k: int = 5) -> float:
    return len(set(retrieved[:k]) & set(relevant)) / k


def recall_at_k(retrieved: list, relevant: list, k: int = 10) -> float:
    if not relevant:
        return 0.0
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant)


def mrr(retrieved: list, relevant: list) -> float:
    rel_set = set(relevant)
    for i, cid in enumerate(retrieved, 1):
        if cid in rel_set:
            return 1.0 / i
    return 0.0


def ndcg_at_k(retrieved: list, scores: dict, k: int = 5) -> float:
    dcg = sum(
        scores.get(cid, 0) / np.log2(i + 1) for i, cid in enumerate(retrieved[:k], 1)
    )
    idcg = sum(
        s / np.log2(i + 1)
        for i, s in enumerate(sorted(scores.values(), reverse=True)[:k], 1)
    )
    return dcg / idcg if idcg > 0 else 0.0


def hit_at_k(retrieved: list, relevant: list, k: int = 5) -> float:
    return 1.0 if set(retrieved[:k]) & set(relevant) else 0.0

Building QDrant retriever and BM25 index

In [11]:

# Load MedEmbed + connect to existing Qdrant collection
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = QdrantVectorStore.from_existing_collection(
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)
qdrant_retriever = vector_store.as_retriever(search_kwargs={"k": QDRANT_TOP})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [12]:
# Build BM25 index from processed_chunks.json using word_tokenize
bm25_docs = [
    Document(
        page_content=chunk.get("page_content", ""),
        metadata={"chunk_id": chunk["metadata"]["chunk_id"],},
    )
    for i, chunk in enumerate(processed_chunks)
]
bm25_retriever = BM25Retriever.from_documents(
    bm25_docs,
    k=BM25_TOP,
    preprocess_func=word_tokenize,
)

print(f"BM25 index built over {len(bm25_docs)} chunks (top-{BM25_TOP})")

BM25 index built over 2632 chunks (top-10)


Reciprocal Rank Fusion via Langchain's EnsembleRetriever

In [13]:
rrf_retriever = EnsembleRetriever(
    retrievers=[qdrant_retriever, bm25_retriever],
    weights=[0.7, 0.3],
)

Query Transformation Function

In [21]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Assuming GROQ_API_KEY is available as a secret or environment variable
# You can set it like: GROQ_API_KEY = "your_groq_api_key_here"
GROQ_API_KEY = ""

llm = ChatGroq(
    temperature=0,
    groq_api_key=GROQ_API_KEY,
    model_name="openai/gpt-oss-20b",
    reasoning_effort='low'
)

def transform_query(query: str, prompt_version) -> str:
  prompt = ChatPromptTemplate.from_messages([
      ("system", prompt_version),
      ("human", "{query}")
  ])

  chain = prompt | llm
  out = (chain.invoke({"query": query}).content or "").strip()
  return out or query

Eval Loop

In [22]:
def evaluate_rrf(prompt_version:str) -> dict:
    """Run MedEmbed + BM25 via RRF → top-5 directly (no reranker)."""
    p5_list, r10_list, mrr_list, ndcg5_list, hit5_list = ([], [], [], [], [])
    qt_lat_list, retrieval_lat_list, total_lat_list = ([], [], [])

    for i, item in enumerate(eval_queries, 1):
        query = item["query"]

        t0 = time.time()
        optimized_query = transform_query(query, prompt_version)
        qt_lat = time.time() - t0

        relevant_ids = item["ground_truth"]["relevant_chunk_ids"]
        rel_scores = item["ground_truth"]["relevance_scores"]

        t1 = time.time()
        # EnsembleRetriever applies RRF internally; take top FINAL_TOP directly
        rrf_docs = rrf_retriever.invoke(optimized_query)
        top_docs = rrf_docs[:FINAL_TOP]
        retrieval_lat = time.time() - t1
        total_lat = qt_lat + retrieval_lat

        # Extract chunk IDs for metric computation
        retrieved_ids = [doc.metadata.get("chunk_id") for doc in top_docs]
        all_rrf_ids = [doc.metadata.get("chunk_id") for doc in rrf_docs]

        p5 = precision_at_k(retrieved_ids, relevant_ids, k=5)
        r10 = recall_at_k(all_rrf_ids, relevant_ids, k=10)
        mrr_s = mrr(retrieved_ids, relevant_ids)
        ndcg5 = ndcg_at_k(retrieved_ids, rel_scores, k=5)
        hit5 = hit_at_k(retrieved_ids, relevant_ids, k=5)

        p5_list.append(p5)
        r10_list.append(r10)
        mrr_list.append(mrr_s)
        ndcg5_list.append(ndcg5)
        hit5_list.append(hit5)
        qt_lat_list.append(qt_lat)
        retrieval_lat_list.append(retrieval_lat)
        total_lat_list.append(total_lat)

        print(
            f"  [{i:02d}] P@5={p5:.2f} NDCG@5={ndcg5:.2f} Hit@5={int(hit5)} "
            f"total_rrf={len(rrf_docs):02d} | QT: {qt_lat * 1000:.0f}ms | Ret: {retrieval_lat * 1000:.0f}ms | Total: {total_lat * 1000:.0f}ms | {query[:50]}…"
        )

    return {
        "Pipeline": "MedEmbed + BM25 RRF [0.7, 0.3] (k=10)",
        "Precision@5": round(float(np.mean(p5_list)), 3),
        "Recall@10": round(float(np.mean(r10_list)), 3),
        "MRR": round(float(np.mean(mrr_list)), 3),
        "NDCG@5": round(float(np.mean(ndcg5_list)), 3),
        "Hit@5": round(float(np.mean(hit5_list)), 3),
        "Query_Transformation_Latency_ms": round(float(np.mean(qt_lat_list)) * 1000, 1),
        "Retrieval_Latency_ms": round(float(np.mean(retrieval_lat_list)) * 1000, 1),
        "Total_Latency_ms": round(float(np.mean(total_lat_list)) * 1000, 1),
    }


#rrf2_result = evaluate_rrf()

In [16]:
#V2 result (with reasoning not low)
rrf2_result

{'Pipeline': 'MedEmbed + BM25 RRF [0.7, 0.3] (k=10)',
 'Precision@5': 0.393,
 'Recall@10': 0.532,
 'MRR': 0.839,
 'NDCG@5': 0.503,
 'Hit@5': 0.933,
 'Query_Transformation_Latency_ms': 4039.3,
 'Retrieval_Latency_ms': 648.0,
 'Total_Latency_ms': 4687.3}

In [33]:
v2result_with_reasoning = rrf2_result

## Running Prompt Version 3

In [17]:
V3 = """Extract a search query from the clinical scenario.

Output one line: the key entities (drugs, conditions, lab values, treatment-relevant patient factors) copied from the scenario, followed by the clinical issue in at most 5 words.
No other words. Do not answer or add information."""

In [23]:
v3_result = evaluate_rrf(V3)

  [01] P@5=0.40 NDCG@5=0.75 Hit@5=1 total_rrf=13 | QT: 285ms | Ret: 1203ms | Total: 1487ms | A 55-year-old patient with advanced breast cancer …
  [02] P@5=0.60 NDCG@5=0.83 Hit@5=1 total_rrf=14 | QT: 485ms | Ret: 410ms | Total: 895ms | What patient education should be provided to an in…
  [03] P@5=0.20 NDCG@5=0.16 Hit@5=1 total_rrf=15 | QT: 161ms | Ret: 322ms | Total: 483ms | A 30-year-old patient with type 1 diabetes, 20 wee…
  [04] P@5=0.60 NDCG@5=0.78 Hit@5=1 total_rrf=18 | QT: 442ms | Ret: 364ms | Total: 806ms | A 45-year-old patient with a history of type 2 dia…
  [05] P@5=0.40 NDCG@5=0.38 Hit@5=1 total_rrf=15 | QT: 390ms | Ret: 320ms | Total: 710ms | An 80-year-old patient with diabetes is being cons…
  [06] P@5=0.00 NDCG@5=0.00 Hit@5=0 total_rrf=18 | QT: 518ms | Ret: 502ms | Total: 1020ms | A 55-year-old patient with type 2 diabetes and sta…
  [07] P@5=0.20 NDCG@5=0.42 Hit@5=1 total_rrf=17 | QT: 280ms | Ret: 534ms | Total: 813ms | A 45-year-old patient with type 2 diabetes and o

In [24]:
v3_result

{'Pipeline': 'MedEmbed + BM25 RRF [0.7, 0.3] (k=10)',
 'Precision@5': 0.353,
 'Recall@10': 0.489,
 'MRR': 0.732,
 'NDCG@5': 0.449,
 'Hit@5': 0.9,
 'Query_Transformation_Latency_ms': 345.6,
 'Retrieval_Latency_ms': 555.1,
 'Total_Latency_ms': 900.7}

## Running Prompt Version 2: Low Reasoning Effort

In [25]:
V2 = """Condense the clinical scenario into one plain-English question that a guideline passage could directly answer.

- Keep every drug, condition, lab value, and number that matters to the decision.
- Drop demographics and story details that do not change management.
- Do not add facts, advice, or new terminology.
- One sentence, under 25 words. Output only the question."""

In [26]:
v2__lowreasoning_result = evaluate_rrf(V2)

  [01] P@5=0.60 NDCG@5=0.95 Hit@5=1 total_rrf=16 | QT: 399ms | Ret: 941ms | Total: 1341ms | A 55-year-old patient with advanced breast cancer …
  [02] P@5=0.40 NDCG@5=0.46 Hit@5=1 total_rrf=16 | QT: 323ms | Ret: 371ms | Total: 694ms | What patient education should be provided to an in…
  [03] P@5=0.60 NDCG@5=0.43 Hit@5=1 total_rrf=16 | QT: 381ms | Ret: 375ms | Total: 756ms | A 30-year-old patient with type 1 diabetes, 20 wee…
  [04] P@5=0.60 NDCG@5=0.78 Hit@5=1 total_rrf=18 | QT: 396ms | Ret: 404ms | Total: 800ms | A 45-year-old patient with a history of type 2 dia…
  [05] P@5=0.60 NDCG@5=0.74 Hit@5=1 total_rrf=17 | QT: 258ms | Ret: 384ms | Total: 642ms | An 80-year-old patient with diabetes is being cons…
  [06] P@5=0.40 NDCG@5=0.29 Hit@5=1 total_rrf=17 | QT: 309ms | Ret: 370ms | Total: 679ms | A 55-year-old patient with type 2 diabetes and sta…
  [07] P@5=0.40 NDCG@5=0.60 Hit@5=1 total_rrf=20 | QT: 272ms | Ret: 340ms | Total: 613ms | A 45-year-old patient with type 2 diabetes and obe

In [27]:
v2__lowreasoning_result

{'Pipeline': 'MedEmbed + BM25 RRF [0.7, 0.3] (k=10)',
 'Precision@5': 0.407,
 'Recall@10': 0.536,
 'MRR': 0.728,
 'NDCG@5': 0.478,
 'Hit@5': 0.9,
 'Query_Transformation_Latency_ms': 361.9,
 'Retrieval_Latency_ms': 443.7,
 'Total_Latency_ms': 805.7}

### Running Prompt Version 1

In [28]:
V1 = """Rewrite the clinical scenario as a short search query for retrieving passages from diabetes clinical guidelines.

Rules:
- Use only words that appear in the scenario. Do not add terms, synonyms, or abbreviation expansions.
- Keep drug names, lab values, numbers, conditions, and patient factors that affect treatment (e.g., pregnancy, kidney disease).
- Remove narrative filler and details that do not affect the decision.
- Keep what is being asked.
- Do not answer. Output one line under 30 words."""

In [34]:
v2result_with_reasoning

{'Pipeline': 'MedEmbed + BM25 RRF [0.7, 0.3] (k=10)',
 'Precision@5': 0.393,
 'Recall@10': 0.532,
 'MRR': 0.839,
 'NDCG@5': 0.503,
 'Hit@5': 0.933,
 'Query_Transformation_Latency_ms': 4039.3,
 'Retrieval_Latency_ms': 648.0,
 'Total_Latency_ms': 4687.3}

In [29]:
v1_result = evaluate_rrf(V1)

  [01] P@5=0.60 NDCG@5=0.96 Hit@5=1 total_rrf=14 | QT: 229ms | Ret: 910ms | Total: 1138ms | A 55-year-old patient with advanced breast cancer …
  [02] P@5=0.80 NDCG@5=0.92 Hit@5=1 total_rrf=13 | QT: 199ms | Ret: 490ms | Total: 689ms | What patient education should be provided to an in…
  [03] P@5=0.20 NDCG@5=0.24 Hit@5=1 total_rrf=14 | QT: 180ms | Ret: 508ms | Total: 688ms | A 30-year-old patient with type 1 diabetes, 20 wee…
  [04] P@5=0.80 NDCG@5=0.83 Hit@5=1 total_rrf=19 | QT: 591ms | Ret: 505ms | Total: 1096ms | A 45-year-old patient with a history of type 2 dia…
  [05] P@5=0.60 NDCG@5=0.69 Hit@5=1 total_rrf=15 | QT: 384ms | Ret: 475ms | Total: 859ms | An 80-year-old patient with diabetes is being cons…
  [06] P@5=0.00 NDCG@5=0.00 Hit@5=0 total_rrf=15 | QT: 520ms | Ret: 515ms | Total: 1035ms | A 55-year-old patient with type 2 diabetes and sta…
  [07] P@5=0.40 NDCG@5=0.69 Hit@5=1 total_rrf=16 | QT: 272ms | Ret: 486ms | Total: 758ms | A 45-year-old patient with type 2 diabetes and o

In [31]:
v1_result

{'Pipeline': 'MedEmbed + BM25 RRF [0.7, 0.3] (k=10)',
 'Precision@5': 0.42,
 'Recall@10': 0.554,
 'MRR': 0.836,
 'NDCG@5': 0.546,
 'Hit@5': 0.933,
 'Query_Transformation_Latency_ms': 325.2,
 'Retrieval_Latency_ms': 568.8,
 'Total_Latency_ms': 894.0}

Evaluated LLM-based query rewriting as a pre-retrieval step. The best prompt (extractive, V1) matched the RRF baseline on P@5 (0.420 vs 0.427) and Recall@10, with a modest MRR gain (0.836 vs 0.768), but added about 325 ms and an additional LLM dependency. Given the small evaluation set (n=30) and no clear precision benefit, we retained the simpler RRF configuration without rewriting. A prompt that added domain terminology degraded all metrics (P@5 0.293, MRR 0.583), consistent with keyword pollution of the BM25 component.

In [35]:
results = {
    "V1": v1_result,
    "V2 (low reasoning)": v2__lowreasoning_result,
    "V2 (with reasoning)": v2result_with_reasoning,
    "V3": v3_result
}

results_df = pd.DataFrame.from_dict(results, orient='index')
display(results_df)

,Pipeline,Precision@5,Recall@10,MRR,NDCG@5,Hit@5,Query_Transformation_Latency_ms,Retrieval_Latency_ms,Total_Latency_ms
V1,"MedEmbed + BM25 RRF [0.7, 0.3] (k=10)",0.420,0.554,0.836,0.546,0.933,325.2,568.8,894.0
V2 (low reasoning),"MedEmbed + BM25 RRF [0.7, 0.3] (k=10)",0.407,0.536,0.728,0.478,0.900,361.9,443.7,805.7
V2 (with reasoning),"MedEmbed + BM25 RRF [0.7, 0.3] (k=10)",0.393,0.532,0.839,0.503,0.933,4039.3,648.0,4687.3
V3,"MedEmbed + BM25 RRF [0.7, 0.3] (k=10)",0.353,0.489,0.732,0.449,0.900,345.6,555.1,900.7
